# 1. GPU
我们可以指定用于存储和计算的设备，如CPU和GPU。
默认情况下，张量是在内存中创建的，然后使用CPU计算它。

在PyTorch中，CPU和GPU可以用`torch.device('cpu')`
和`torch.device('cuda')`表示。
应该注意的是，`cpu`设备意味着所有物理CPU和内存，
这意味着PyTorch的计算将尝试使用所有CPU核心。
然而，`gpu`设备只代表一个卡和相应的显存。
如果有多个GPU，我们使用`torch.device(f'cuda:{i}')`
来表示第$i$块GPU（$i$从0开始）。
另外，`cuda:0`和`cuda`是等价的。

In [1]:
import torch
from torch import nn
torch.device('cpu'), torch.device('cuda')

(device(type='cpu'), device(type='cuda'))

查询可用gpu的数量

In [2]:
torch.cuda.device_count()

1

In [3]:
def try_gpu(i=0):  #@save
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():  #@save
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]"""
    devices = [torch.device(f'cuda:{i}')
             for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

## 1.1 张量与GPU
查询张量所在的设备,默认情况下，张量是在cpu上创建的

In [4]:
x = torch.tensor([1, 2, 3])
print(x.device)

cpu


需要注意的是，无论何时我们要对多个项进行操作，
它们都**必须在同一个设备**上。
例如，如果我们对两个张量求和，
我们需要确保两个张量都位于同一个设备上，
否则框架将不知道在哪里存储结果，甚至不知道在哪里执行计算。

### 存储在GPU上

有几种方法可以在GPU上存储张量。
例如，我们可以在创建张量时指定存储设备。接下来，我们在第一个`gpu`上创建张量变量`X`。
在GPU上创建的张量只消耗这个GPU的显存。
我们可以使用`nvidia-smi`命令查看显存使用情况。
一般来说，我们需要确保不创建超过GPU显存限制的数据。

In [5]:
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [6]:
!nvidia-smi

Sun Jul 26 08:16:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 573.57                 Driver Version: 573.57         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   64C    P8              4W /   35W |     438MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

将张量复制 / 迁移到 指定的 GPU 设备
```python
Z = X.cuda(1)
```

人们使用GPU来进行机器学习，因为单个GPU相对运行速度快。
但是在设备（CPU、GPU和其他机器）之间传输数据比计算慢得多。
这也使得并行化变得更加困难，因为我们必须等待数据被发送（或者接收），
然后才能继续进行更多的操作。
这就是为什么拷贝操作要格外小心。
根据经验，多个小操作比一个大操作糟糕得多。
此外，一次执行几个操作比代码中散布的许多单个操作要好得多。
如果一个设备必须等待另一个设备才能执行其他操作，
那么这样的操作可能会阻塞。
这有点像排队订购咖啡，而不像通过电话预先订购：
当客人到店的时候，咖啡已经准备好了。

最后，当我们打印张量或将张量转换为NumPy格式时，
如果数据不在内存中，框架会首先将其复制到内存中，
这会导致额外的传输开销。
更糟糕的是，它现在受制于全局解释器锁，使得一切都得等待Python完成。

## 1.2 神经网络与GPU

类似地，神经网络模型可以指定设备。
下面的代码将模型参数放在GPU上。

In [7]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())

在接下来的几章中，
我们将看到更多关于如何在GPU上运行模型的例子，
因为它们将变得更加计算密集。

当输入为GPU上的张量时，模型将在同一GPU上计算结果。

In [8]:
net(X)

tensor([[-1.1201],
        [-1.1201]], device='cuda:0', grad_fn=<AddmmBackward0>)

* 不经意地移动数据可能会显著降低性能。一个典型的错误如下：计算GPU上每个小批量的损失，并在命令行中将其报告给用户（或将其记录在NumPy `ndarray`中）时，将触发全局解释器锁，从而使所有GPU阻塞。最好是为GPU内部的日志分配内存，并且只移动较大的日志。

In [9]:
import time

device = torch.device("cuda")
n = 100
num_iter = 1000

# 计时开始
start = time.time()
record = []

for _ in range(num_iter):
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    C = A @ B
    # ❌ 每一轮直接搬到CPU，逐个存入列表（题目指定的糟糕写法）
    norm_val = torch.norm(C).cpu().item()
    record.append(norm_val)

end = time.time()
print(f"总耗时：{end - start:.4f} 秒")
print(f"记录数量：{len(record)}")

总耗时：0.2849 秒
记录数量：1000


In [10]:
device = torch.device("cuda")
n = 100
num_iter = 1000

start = time.time()
gpu_buffer = []

for _ in range(num_iter):
    A = torch.randn(n, n, device=device)
    B = torch.randn(n, n, device=device)
    C = A @ B
    # 先保存在GPU显存，不往CPU搬
    gpu_buffer.append(torch.norm(C))

# ✅循环结束后一次性转移到CPU
record = torch.stack(gpu_buffer).cpu().tolist()

end = time.time()
print(f"总耗时：{end - start:.4f} 秒")
print(f"记录数量：{len(record)}")

总耗时：0.1625 秒
记录数量：1000
